# 02 - Traitement des Données Texte

## Section : Gestion du Texte

## Objectif
Transformer les données brutes en un dataset prêt pour la modélisation en :
- Nettoyant le HTML des descriptions
- Gérant les valeurs manquantes (35% de descriptions vides)
- Normalisant les textes
- Créant des features supplémentaires
- Préparant le dataset final pour la modélisation

## Plan
1. Chargement des données
2. Phase 1 : Nettoyage HTML
3. Phase 2 : Gestion des valeurs manquantes
4. Phase 3 : Normalisation des textes
5. Phase 4 : Feature Engineering
6. Phase 5 : Préparation finale et sauvegarde
   - 6.2 bis. Superclasse Publications (optionnel, à tester)



In [ ]:
# Import des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Racine du projet : chercher le dossier contenant 'src'
current = Path.cwd()
PROJECT_ROOT = current
while not (PROJECT_ROOT / 'src').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        PROJECT_ROOT = current  # fallback
        break
    PROJECT_ROOT = parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import des modules de preprocessing
from src.utils.data_loader import load_data, save_data
from src.preprocessing import (
    clean_html,
    decode_html_entities,
    has_html,
    remove_remaining_html,
    normalize_text,
    combine_texts,
    create_length_features,
    create_binary_features,
    create_quality_features,
    PreprocessingPipeline
)

# Configuration des graphiques
try:
    plt.style.use('seaborn-v0_8')
except (OSError, ValueError):
    plt.style.use('ggplot')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Bibliothèques importées avec succès")



## 1. Chargement des Données

Chargement des données brutes depuis le dossier `data brut/`.


In [ ]:
# Chargement des données
DATA_DIR = PROJECT_ROOT / 'data brut'

X_train, y_train, X_test = load_data(data_dir=DATA_DIR)

# Vérification de l'alignement
if X_train.shape[0] == y_train.shape[0]:
    print(f"\n✅ Alignement OK : X_train et y_train ont le même nombre de lignes")
else:
    print(f"\n⚠️  ATTENTION : X_train et y_train n'ont pas le même nombre de lignes !")

# Aperçu des données
print("\n" + "="*80)
print("APERÇU DE X_train (avant preprocessing)")
print("="*80)
print(X_train.head())
print(f"\nValeurs manquantes :")
print(X_train.isnull().sum())



## 2. Phase 1 : Nettoyage HTML

### 2.1 Analyse du HTML avant nettoyage


In [ ]:
# Détection du HTML dans les descriptions
X_train['has_html_before'] = X_train['description'].apply(has_html)
X_test['has_html_before'] = X_test['description'].apply(has_html)

html_count_train = X_train['has_html_before'].sum()
html_count_test = X_test['has_html_before'].sum()

print("📊 Présence de HTML dans les descriptions (AVANT nettoyage) :")
print(f"  - X_train : {html_count_train} descriptions avec HTML ({html_count_train/len(X_train)*100:.2f}%)")
print(f"  - X_test : {html_count_test} descriptions avec HTML ({html_count_test/len(X_test)*100:.2f}%)")

# Exemples de descriptions avec HTML (textes courts, affichage intégral)
print("\n📝 Exemples de descriptions avec HTML (textes courts, affichage intégral) :")
html_mask = X_train['has_html_before']
short_mask = X_train['description'].fillna('').str.len() <= 300
html_examples = X_train.loc[html_mask & short_mask, 'description'].head(3)
for idx, desc in enumerate(html_examples, 1):
    print(f"\nExemple {idx} :")
    print(desc)



### 2.2 Nettoyage du HTML


In [ ]:
# Nettoyage HTML des descriptions
print("🔄 Nettoyage du HTML...")

X_train['description_clean'] = X_train['description'].apply(
    lambda x: clean_html(x, preserve_structure=True)
)
X_test['description_clean'] = X_test['description'].apply(
    lambda x: clean_html(x, preserve_structure=True)
)

# Vérification après nettoyage
X_train['has_html_after'] = X_train['description_clean'].apply(has_html)
X_test['has_html_after'] = X_test['description_clean'].apply(has_html)

html_remaining_train = X_train['has_html_after'].sum()
html_remaining_test = X_test['has_html_after'].sum()

print(f"✅ Nettoyage terminé !")
print(f"  - X_train : {html_remaining_train} descriptions avec HTML restant ({html_remaining_train/len(X_train)*100:.2f}%)")
print(f"  - X_test : {html_remaining_test} descriptions avec HTML restant ({html_remaining_test/len(X_test)*100:.2f}%)")

# Comparaison longueurs avant/après
X_train['desc_length_before'] = X_train['description'].fillna('').astype(str).str.len()
X_train['desc_length_after'] = X_train['description_clean'].str.len()

print(f"\n📊 Comparaison des longueurs (X_train) :")
print(f"  - Avant nettoyage : moyenne = {X_train['desc_length_before'].mean():.1f}, médiane = {X_train['desc_length_before'].median():.1f}")
print(f"  - Après nettoyage : moyenne = {X_train['desc_length_after'].mean():.1f}, médiane = {X_train['desc_length_after'].median():.1f}")

# Exemples après nettoyage (textes courts, affichage intégral)
print("\n📝 Exemples de descriptions nettoyées (textes courts, affichage intégral) :")
html_mask = X_train['has_html_before']
short_mask = X_train['description'].fillna('').str.len() <= 300
subset = X_train.loc[html_mask & short_mask, ['description', 'description_clean']].head(3)
for i, (before, after) in enumerate(zip(subset['description'], subset['description_clean']), 1):
    print(f"\nExemple {i} :")
    print(f"  AVANT : {before}")
    print(f"  APRÈS : {after}")



## 3. Phase 2 : Gestion des Valeurs Manquantes

### 3.1 Analyse des valeurs manquantes


In [ ]:
# Analyse des valeurs manquantes
missing_train = X_train['description_clean'].isna() | (X_train['description_clean'] == '')
missing_test = X_test['description_clean'].isna() | (X_test['description_clean'] == '')

missing_count_train = missing_train.sum()
missing_count_test = missing_test.sum()

print("📊 Analyse des valeurs manquantes (descriptions) :")
print(f"  - X_train : {missing_count_train} descriptions vides ({missing_count_train/len(X_train)*100:.2f}%)")
print(f"  - X_test : {missing_count_test} descriptions vides ({missing_count_test/len(X_test)*100:.2f}%)")

# Analyse par classe (si y_train disponible)
if 'prdtypecode' not in X_train.columns:
    # Fusionner avec y_train pour analyser par classe
    df_train_merged = pd.concat([X_train, y_train], axis=1)
else:
    df_train_merged = X_train.copy()

if 'prdtypecode' in df_train_merged.columns:
    missing_by_class = df_train_merged.groupby('prdtypecode').agg({
        'description_clean': lambda x: (x.isna() | (x == '')).sum() / len(x) * 100
    }).round(2).sort_values('description_clean', ascending=False)
    
    print(f"\n📊 Top 10 classes avec le plus de descriptions manquantes :")
    print(missing_by_class.head(10))



### 3.2 Application de la stratégie de gestion des valeurs manquantes

**Stratégie choisie :** `designation_only` - Pour les produits sans description, on utilisera uniquement la designation.


In [ ]:
# Les descriptions vides sont déjà gérées (chaîne vide)
# On s'assure qu'elles sont bien des chaînes vides et non NaN
X_train['description_clean'] = X_train['description_clean'].fillna('')
X_test['description_clean'] = X_test['description_clean'].fillna('')

print("✅ Valeurs manquantes gérées : descriptions vides remplacées par chaîne vide")
print(f"  - X_train : {X_train['description_clean'].isna().sum()} NaN restants")
print(f"  - X_test : {X_test['description_clean'].isna().sum()} NaN restants")



## 4. Phase 3 : Normalisation des Textes

Normalisation des désignations et descriptions nettoyées.


In [ ]:
# Normalisation des textes
print("🔄 Normalisation des textes...")

# Normalisation des désignations
X_train['designation_clean'] = X_train['designation'].apply(
    lambda x: normalize_text(x, lowercase=True, remove_extra_spaces=True, remove_accents=False)
)
X_test['designation_clean'] = X_test['designation'].apply(
    lambda x: normalize_text(x, lowercase=True, remove_extra_spaces=True, remove_accents=False)
)

# Normalisation des descriptions (déjà nettoyées du HTML)
X_train['description_clean'] = X_train['description_clean'].apply(
    lambda x: normalize_text(x, lowercase=True, remove_extra_spaces=True, remove_accents=False)
)
X_test['description_clean'] = X_test['description_clean'].apply(
    lambda x: normalize_text(x, lowercase=True, remove_extra_spaces=True, remove_accents=False)
)

# Passe de sécurité : supprimer tout HTML restant (malformé : < li>, <br/>, <. 4pc br>, etc.)
X_train['description_clean'] = X_train['description_clean'].apply(remove_remaining_html)
X_test['description_clean'] = X_test['description_clean'].apply(remove_remaining_html)

print("✅ Normalisation terminée !")

# Exemples
print("\n📝 Exemples de textes normalisés :")
for idx in range(3):
    print(f"\nExemple {idx+1} :")
    print(f"  Designation originale : {X_train['designation'].iloc[idx][:100]}...")
    print(f"  Designation normalisée : {X_train['designation_clean'].iloc[idx][:100]}...")



## 5. Phase 4 : Feature Engineering

Création de features supplémentaires pour améliorer la classification.


In [ ]:
# Création de toutes les features
print("🔄 Création des features...")

# Utiliser les colonnes originales pour les features de longueur
X_train_features = X_train[['designation', 'description']].copy()
X_test_features = X_test[['designation', 'description']].copy()

# Créer les features
X_train_features = create_length_features(X_train_features)
X_train_features = create_binary_features(X_train_features)
X_train_features = create_quality_features(X_train_features)

X_test_features = create_length_features(X_test_features)
X_test_features = create_binary_features(X_test_features)
X_test_features = create_quality_features(X_test_features)

# Ajouter les features au DataFrame principal
feature_cols = [col for col in X_train_features.columns if col not in ['designation', 'description']]
for col in feature_cols:
    X_train[col] = X_train_features[col]
    X_test[col] = X_test_features[col]

print(f"✅ {len(feature_cols)} features créées :")
for col in feature_cols:
    print(f"  - {col}")

# Aperçu des features
print("\n📊 Aperçu des features créées :")
print(X_train[feature_cols].describe())



## 6. Phase 5 : Préparation Finale

### 6.1 Combinaison des textes


In [ ]:
# Combiner designation et description en un seul texte
print("🔄 Combinaison des textes...")

X_train['text_combined'] = X_train.apply(
    lambda row: combine_texts(
        row['designation_clean'],
        row['description_clean'],
        separator=' ',
        handle_missing='designation_only'
    ),
    axis=1
)

X_test['text_combined'] = X_test.apply(
    lambda row: combine_texts(
        row['designation_clean'],
        row['description_clean'],
        separator=' ',
        handle_missing='designation_only'
    ),
    axis=1
)

print("✅ Textes combinés !")

# Statistiques sur les textes combinés
print(f"\n📊 Statistiques sur les textes combinés (X_train) :")
print(f"  - Longueur moyenne : {X_train['text_combined'].str.len().mean():.1f} caractères")
print(f"  - Longueur médiane : {X_train['text_combined'].str.len().median():.1f} caractères")
print(f"  - Longueur min : {X_train['text_combined'].str.len().min()} caractères")
print(f"  - Longueur max : {X_train['text_combined'].str.len().max()} caractères")

# Vérifier qu'il n'y a pas de textes vides
empty_texts = (X_train['text_combined'] == '').sum()
print(f"  - Textes vides : {empty_texts} ({empty_texts/len(X_train)*100:.2f}%)")



### 6.2 Préparation des datasets finaux


In [ ]:
# Sélectionner les colonnes importantes pour le dataset final
columns_to_keep = [
    'productid',
    'imageid',
    'designation_clean',
    'description_clean',
    'text_combined',
    'designation_length',
    'description_length',
    'total_text_length',
    'designation_word_count',
    'description_word_count',
    'total_word_count',
    'has_description',
    'has_html',
    'is_description_empty',
    'text_completeness',
    'description_quality_score',
    'word_density'
]

# Créer les datasets finaux
X_train_final = X_train[columns_to_keep].copy()
X_test_final = X_test[columns_to_keep].copy()

print("✅ Datasets finaux préparés !")
print(f"\n📊 Structure des datasets finaux :")
print(f"  - X_train_final : {X_train_final.shape[0]:,} lignes × {X_train_final.shape[1]} colonnes")
print(f"  - X_test_final : {X_test_final.shape[0]:,} lignes × {X_test_final.shape[1]} colonnes")
print(f"\nColonnes :")
for col in columns_to_keep:
    print(f"  - {col}")



### 6.3 Sauvegarde des datasets nettoyés


In [ ]:
# Sauvegarder les datasets nettoyés
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("💾 Sauvegarde des datasets...")

# Sauvegarder les datasets finaux
save_data(X_train_final, OUTPUT_DIR / 'X_train_clean.csv')
save_data(X_test_final, OUTPUT_DIR / 'X_test_clean.csv')
save_data(y_train, OUTPUT_DIR / 'y_train.csv')

# Créer y_train_superclass si besoin (au cas où la cellule 6.2 bis n'a pas été exécutée)
if 'y_train_superclass' not in locals():
    PUBLICATION_CLASSES = {10, 2280, 2403, 2705}
    CODE_SUPERCLASSE = 9999
    y_train_superclass = y_train.copy()
    y_train_superclass['prdtypecode'] = y_train['prdtypecode'].apply(
        lambda x: CODE_SUPERCLASSE if x in PUBLICATION_CLASSES else x
    )

save_data(y_train_superclass, OUTPUT_DIR / 'y_train_superclass.csv')

print("\n✅ Datasets sauvegardés dans :", OUTPUT_DIR)
print("   - y_train.csv : 27 classes (référence)")
print("   - y_train_superclass.csv : 24 classes (Publications fusionnées, à tester)")



### 6.2 bis. Création de la superclasse Publications (à tester)

**⚠️ Justification :** Nous avons un doute sur la pertinence du regroupement des 4 classes 
"publications" (10-Livres, 2280-Presse, 2403-BD/Partitions, 2705-Romans). Ces classes partagent 
un vocabulaire très similaire et présentent des confusions importantes en modélisation. 
**Nous devons tester** si une fusion améliore les performances. Une version avec superclasse 
est donc créée en parallèle ; les modèles seront entraînés sur les deux versions pour comparaison.

In [ ]:
# Classes à fusionner en superclasse "Publications" (10, 2280, 2403, 2705)
PUBLICATION_CLASSES = {10, 2280, 2403, 2705}
CODE_SUPERCLASSE = 9999  # Code unique pour la superclasse Publications

# Créer y_train avec superclasse (les 4 classes publications -> 9999)
y_train_superclass = y_train.copy()
y_train_superclass['prdtypecode'] = y_train['prdtypecode'].apply(
    lambda x: CODE_SUPERCLASSE if x in PUBLICATION_CLASSES else x
)

print("📊 Superclasse Publications créée :")
print(f"   - Classes fusionnées : {sorted(PUBLICATION_CLASSES)} (Livres, Presse, BD/Partitions, Romans)")
print(f"   - Nouveau code : {CODE_SUPERCLASSE}")
print(f"   - Avant : {y_train['prdtypecode'].nunique()} classes")
print(f"   - Après : {y_train_superclass['prdtypecode'].nunique()} classes")
print(f"   - Effectif superclasse : {(y_train_superclass['prdtypecode'] == CODE_SUPERCLASSE).sum():,} produits")